### TF-IDF

#### 0. The problem TF-IDF solves

The simplest way to turn text into numbers is a **bag-of-words** count: one
column per vocabulary word, value = how many times that word appears in the
document. Problem: this treats every word as equally important. A word like
"the" appears constantly in every document and tells you nothing about what
that document is *about*, but a raw count would rank it as the most
"important" word in almost any English document, just by frequency.

TF-IDF fixes this with one idea: **a word matters to a document if it's
frequent in that document AND rare across the rest of the corpus.** Frequent
+ rare-elsewhere = distinctive. Frequent + common-everywhere = filler.

Worked corpus we'll use throughout this notebook — 3 tiny documents:
```
doc1: "the cat sat on the mat"
doc2: "the dog sat on the log"
doc3: "cats and dogs are great pets"
```
Notice "the" appears in doc1 and doc2 (common), while "cat"/"mat" appear only
in doc1 (distinctive to it). TF-IDF should end up ranking "cat"/"mat" as more
important to doc1 than "the" is — even though "the" appears MORE often within
doc1 itself. Keep that specific comparison in mind; we'll verify it numerically below.


#### 1. Term Frequency (TF)

Formula: `tf(t, d) = count(t in d) / total_words(d)` — normalized by document
length, so a long document doesn't automatically get bigger TF values just
from having more words overall.

Worked example, doc1 = "the cat sat on the mat" (6 words total):
```
word counts: the=2, cat=1, sat=1, on=1, mat=1
tf(the, doc1) = 2/6 = 0.333
tf(cat, doc1) = 1/6 = 0.167
tf(sat, doc1) = 1/6 = 0.167
tf(on,  doc1) = 1/6 = 0.167
tf(mat, doc1) = 1/6 = 0.167
```
Notice: by raw TF alone, "the" already looks twice as important as any other
word in doc1 (0.333 vs 0.167) — this is the problem TF-IDF's IDF term exists
to correct. TF alone can't distinguish "the" from "cat"; both are just
"a word that appears in this document."


In [ ]:
from collections import Counter

corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
]
docs = [d.split() for d in corpus]

def tf(word, doc):
    counts = Counter(doc)
    return counts[word] / len(doc)

# reproduce the hand-worked doc1 numbers
for word in ["the", "cat", "sat", "on", "mat"]:
    print(f"tf({word}, doc1) = {tf(word, docs[0]):.3f}")

#### 2. Inverse Document Frequency (IDF)

Formula: `idf(t) = log(N / df(t))` — N = total documents in the corpus,
df(t) = number of documents containing term t at least once (not how many
times, just whether it appears at all).

Worked example, N=3 documents:
```
"the": appears in doc1, doc2 → df=2 → idf = log(3/2) = log(1.5) ≈ 0.405
"cat": appears in doc1 only  → df=1 → idf = log(3/1) = log(3)   ≈ 1.099
"sat": appears in doc1, doc2 → df=2 → idf ≈ 0.405
"on":  appears in doc1, doc2 → df=2 → idf ≈ 0.405
"mat": appears in doc1 only  → df=1 → idf ≈ 1.099
```
Already visible: "the" gets a LOWER idf (0.405) than "cat"/"mat" (1.099) —
common-across-documents words get penalized, rare-and-specific words get
boosted. This is the correction TF alone couldn't provide.

Why the log, specifically — without it, IDF would be a raw ratio `N/df(t)`.
For a word appearing in just 1 of 1000 documents, that ratio is 1000, while
a word in 500 of 1000 is only 2 — the rare word would dominate the score by
a factor of 500, wildly overweighting extremely rare (sometimes just noisy
or misspelled) terms. The log compresses that: log(1000)≈6.9 vs log(2)≈0.7,
a ratio of ~10x instead of 500x — rare terms still get more weight, but not
absurdly more.


In [ ]:
import math

def document_frequency(word, all_docs):
    return sum(1 for doc in all_docs if word in doc)

def idf(word, all_docs):
    N = len(all_docs)
    df = document_frequency(word, all_docs)
    return math.log(N / df)

for word in ["the", "cat", "sat", "on", "mat"]:
    print(f"idf({word}) = {idf(word, docs):.3f}")

#### 3. Combining TF × IDF — the payoff

Formula: `tfidf(t, d) = tf(t, d) × idf(t)`

Worked example, doc1's full vector (multiplying the TF values from section 1
by the IDF values from section 2):
```
tfidf(the, doc1) = 0.333 × 0.405 = 0.135
tfidf(cat, doc1) = 0.167 × 1.099 = 0.183
tfidf(sat, doc1) = 0.167 × 0.405 = 0.068
tfidf(on,  doc1) = 0.167 × 0.405 = 0.068
tfidf(mat, doc1) = 0.167 × 1.099 = 0.183
```
**This is the exact thing we set out to verify in section 0**: "the" had the
HIGHEST raw term frequency in doc1 (0.333, twice any other word), but ends up
with a LOWER final TF-IDF score (0.135) than "cat" or "mat" (0.183) — because
"the" appears everywhere (low IDF), while "cat"/"mat" are specific to this
document (high IDF). The combination correctly demotes the frequent-but-
uninformative word and promotes the rarer-but-distinctive ones, exactly the
behavior a raw bag-of-words count could never produce.


In [ ]:
def tfidf(word, doc, all_docs):
    return tf(word, doc) * idf(word, all_docs)

doc1_vector = {word: tfidf(word, docs[0], docs) for word in set(docs[0])}
for word, score in sorted(doc1_vector.items(), key=lambda x: -x[1]):
    print(f"tfidf({word}, doc1) = {score:.3f}")

#### 4. L2 normalization

Why: a longer document naturally accumulates a bigger raw TF-IDF vector
(more nonzero entries, more total "mass") even if it's not actually more
relevant to any given topic — without correcting for this, comparing two
documents' vectors (e.g. via dot product / cosine similarity) would be
biased toward whichever one is simply longer. L2-normalizing divides each
document's vector by its own magnitude, so every document vector has length
1 — comparisons become about the *direction* (which words matter, in what
proportion) rather than raw magnitude.

Formula: `v_normalized = v / ||v||₂`, where `||v||₂ = √(Σ vᵢ²)`

Worked example, doc1's vector from section 3 (only nonzero entries):
```
v = [the=0.135, cat=0.183, sat=0.068, on=0.068, mat=0.183]

||v||₂ = √(0.135² + 0.183² + 0.068² + 0.068² + 0.183²)
       = √(0.0182 + 0.0335 + 0.0046 + 0.0046 + 0.0335)
       = √0.0944 ≈ 0.307

normalized:
  the = 0.135/0.307 ≈ 0.439
  cat = 0.183/0.307 ≈ 0.596
  sat = 0.068/0.307 ≈ 0.221
  on  = 0.068/0.307 ≈ 0.221
  mat = 0.183/0.307 ≈ 0.596
```
Check: √(0.439² + 0.596² + 0.221² + 0.221² + 0.596²) ≈ √(0.193+0.355+0.049+0.049+0.355) = √1.0 = 1.0 ✓
The relative ordering (cat/mat highest, the middling, sat/on lowest) is
unchanged by normalization — only the overall scale shifts, which is exactly
the point: normalization doesn't change *what* the vector says about the
document, just puts it on a comparable footing with every other document's vector.


In [ ]:
import numpy as np

def l2_normalize(vector_dict):
    values = np.array(list(vector_dict.values()))
    norm = np.sqrt(np.sum(values ** 2))
    return {k: v / norm for k, v in vector_dict.items()}

doc1_normalized = l2_normalize(doc1_vector)
for word, score in sorted(doc1_normalized.items(), key=lambda x: -x[1]):
    print(f"{word}: {score:.3f}")

# sanity check: normalized vector's own L2 norm should be 1.0
check = np.sqrt(sum(v**2 for v in doc1_normalized.values()))
print("\nnorm of normalized vector:", check)

#### 5. Comparing against sklearn's `TfidfVectorizer`

Important nuance: sklearn's default IDF formula is NOT the raw
`log(N/df)` used above — it's **smoothed**: `idf(t) = log((1+N)/(1+df(t))) + 1`.
The `+1` inside the log prevents division issues and avoids any term ever
reaching `df=N` and getting idf=0 (which raw log(N/df) would give a word
appearing in every document — technically correct, but sklearn treats
"appears everywhere" as still worth a small non-zero weight, not zero).
The trailing `+1` outside the log shifts every score up slightly.

Because of this, sklearn's numbers won't match our from-scratch numbers
exactly — that's expected, not a bug. What SHOULD match: the *relative
ordering* (cat/mat still outrank "the" after normalization) and the general
shape of the result.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(), index=["doc1", "doc2", "doc3"])
print(df.round(3))

print("\ndoc1, sorted by score (compare ordering to our from-scratch result):")
print(df.loc["doc1"].sort_values(ascending=False).head(5))

#### 6. Limitations and when to use TF-IDF vs. embeddings

- Ignores word order entirely — "dog bites man" and "man bites dog" produce
  the identical bag-of-words vector, despite opposite meanings.
- No notion of synonymy or semantic similarity — "car" and "automobile" are
  completely different columns, zero shared signal, even though they mean
  nearly the same thing. Dense embeddings (word2vec, sentence transformers)
  solve this by placing semantically similar words near each other in
  vector space; TF-IDF has no such geometry.
- Vocabulary is fixed at fit time — a word never seen during `fit` simply
  contributes nothing at inference (see the fraud project's baseline cell
  for exactly this: `vectorizer.transform(val)` uses train's vocabulary only).
- Where it still wins: interpretability (every dimension is literally a
  word, you can read off exactly which words drove a prediction), no
  training required beyond counting, and strong performance specifically
  when the task is close to keyword matching — the fraud project's own
  result (logreg + TF-IDF beat every tree-based model, including hybrid
  LLM-feature setups) is a real example of TF-IDF's strength on a task
  where vocabulary alone is highly discriminative.
